In [1]:
import torch
from torch_geometric.datasets import TUDataset
from torch_geometric.utils import to_networkx
import networkx as nx
import collections
import numpy as np
import pandas as pd

In [2]:
def weisfeiler_lehman_hash(graph):
    colors = {node: data.get('label', 0) for node, data in graph.nodes(data=True)}

    for _ in range(len(graph.nodes())): # Iterate enough times for convergence
        new_colors = {}
        for node in graph.nodes():
            neighbor_colors = sorted([colors[nbr] for nbr in graph.neighbors(node)])
            signature = (colors[node], tuple(neighbor_colors))
            new_colors[node] = hash(signature) # Hash the signature to get a new color

        if new_colors == colors:
            break
        colors = new_colors

    canonical_hash = str(sorted(colors.values()))
    return canonical_hash

In [3]:
def find_isomorphic_groups_from_pyg(dataset):
    hashes = collections.defaultdict(list)

    for i, data in enumerate(dataset):
        # g = to_networkx(data, node_attrs=['x'])
        g = to_networkx(data, node_attrs=['x'], to_undirected=True)
        g.remove_edges_from(nx.selfloop_edges(g))
        bc = nx.betweenness_centrality(g, normalized=False)   # raw counts
        bc_int = {u: int(round(bc[u])) for u in g.nodes()}    # make sure integer
        nx.set_node_attributes(g, bc_int, 'label')
        if len(g) > 0:
            h = weisfeiler_lehman_hash(g)
            hashes[h].append(i)
            
    isomorphic_groups = {h: indices for h, indices in hashes.items() if len(indices) > 1}
    return isomorphic_groups

In [4]:
print("1. Loading AIDS dataset using PyTorch Geometric...")
try:
        # use_node_attr=True loads the node labels into the `x` attribute
    dataset = TUDataset(root='/tmp/AIDS', name='AIDS', use_node_attr=True)
    print(f"   Successfully loaded {len(dataset)} graphs.")
except Exception as e:
    print(f"Error: Could not download or load the dataset. {e}")
    print("Please check your internet connection and if the TUDataset repository is accessible.")
    exit()

print("\n2. Running Weisfeiler-Lehman isomorphism test...")
isomorphic_groups = find_isomorphic_groups_from_pyg(dataset)
print("   Test complete.")

print("\n3. Isomorphism Statistics:")
if not isomorphic_groups:
    print("   No isomorphic graphs were found in the dataset.")
else:
    num_isomorphic_groups = len(isomorphic_groups)
    total_graphs_in_groups = sum(len(indices) for indices in isomorphic_groups.values())
    print(f"   - Total Isomorphic Groups Found: {num_isomorphic_groups}")
    print(f"   - Total Graphs in Isomorphic Groups: {total_graphs_in_groups}")

    print("\n   Isomorphic Group Details (Graph Indices):")
        # Sort groups by the first graph index for consistent output
    sorted_groups = sorted(isomorphic_groups.values(), key=lambda x: x[0])
    for i, group in enumerate(sorted_groups):
        print(f"     - Group {i+1}: {group}")

1. Loading AIDS dataset using PyTorch Geometric...
   Successfully loaded 2000 graphs.

2. Running Weisfeiler-Lehman isomorphism test...
   Test complete.

3. Isomorphism Statistics:
   - Total Isomorphic Groups Found: 353
   - Total Graphs in Isomorphic Groups: 1175

   Isomorphic Group Details (Graph Indices):
     - Group 1: [3, 217, 554, 556, 880, 1006, 1089, 1321, 1605]
     - Group 2: [6, 971]
     - Group 3: [7, 1160, 1539]
     - Group 4: [9, 846, 927]
     - Group 5: [10, 1555]
     - Group 6: [11, 82, 1772]
     - Group 7: [13, 122, 669, 1732, 1838]
     - Group 8: [14, 286]
     - Group 9: [17, 1013, 1395]
     - Group 10: [20, 110, 408, 465, 652, 960, 1187, 1219, 1312, 1461, 1778]
     - Group 11: [22, 304, 886, 1325, 1725, 1922]
     - Group 12: [23, 1950]
     - Group 13: [24, 1966]
     - Group 14: [25, 302]
     - Group 15: [26, 1184]
     - Group 16: [27, 187, 230, 574, 603, 666]
     - Group 17: [28, 705, 973, 1215]
     - Group 18: [29, 1485, 1494, 1804]
     - Group

In [5]:
import random
import networkx as nx
from itertools import combinations

def random_single_edge_perturbation(G: nx.Graph):
    """
    Randomly EITHER add one non-existent edge OR remove one existing edge.
    Exactly one change is made.
    Returns (H, op, edge).
    """
    H = G.copy()
    nodes = list(H.nodes())
    edges = {tuple(sorted(e)) for e in H.edges()}
    nonedges = [e for e in combinations(nodes, 2) if e not in edges]

    can_add = len(nonedges) > 0
    can_remove = len(edges) > 0

    if can_add and can_remove:
        op = random.choice(["add", "remove"])
    elif can_add:
        op = "add"
    elif can_remove:
        op = "remove"

    if op == "add":
        e = random.choice(nonedges)
        H.add_edge(*e)
    else:
        e = random.choice(list(edges))
        H.remove_edge(*e)

    # sanity check: must differ by exactly one edge
    E1 = {tuple(sorted(ed)) for ed in G.edges()}
    E2 = {tuple(sorted(ed)) for ed in H.edges()}
    diff = E1 ^ E2
    if len(diff) != 1:
        raise ValueError("Perturbation failed: more than one change detected.")

    return H, op, e


In [6]:
def relabel_with_bc_int(G):
    H = G.copy()
    bc = nx.betweenness_centrality(H, normalized=False)
    nx.set_node_attributes(H, {v: int(round(bc[v])) for v in H.nodes()}, "label")
    return H

def relabel_with_degree(G):
    H = G.copy()
    nx.set_node_attributes(H, dict(H.degree()), "label")
    return H

def relabel_with_atom(data):
    G = to_networkx(data, node_attrs=['x'], to_undirected=True)
    G.remove_edges_from(nx.selfloop_edges(G))
    atom_labels = {i: int(np.argmax(data.x[i].numpy())) for i in range(data.num_nodes)}
    nx.set_node_attributes(G, atom_labels, "label")
    return G

In [9]:
def extract_largest_group(isomorphic_groups):
    """Return (group_hash, [indices]) for the largest WL group."""
    group_hash = max(isomorphic_groups, key=lambda h: len(isomorphic_groups[h]))
    return group_hash, isomorphic_groups[group_hash]

In [10]:
def run_experiment_A(label_mode, group_indices, dataset, copies_per_graph=10, csv_path=None):
    print(f"\n=== Running Experiment A with labels: {label_mode} ===")

    # 1) Build perturbed copies
    copies = []
    for idx in group_indices:
        G0 = to_networkx(dataset[idx], to_undirected=True)
        G0.remove_edges_from(nx.selfloop_edges(G0))
        for j in range(copies_per_graph):
            H, op, e = random_single_edge_perturbation(G0)
            copies.append({"orig_idx": idx, "copy_idx": j, "op": op, "edge": e, "graph": H})

    print(f"Total perturbed copies: {len(copies)}")

    # 2) Relabel + WL
    for d in copies:
        if label_mode == "atom":
            H_labeled = relabel_with_atom(dataset[d["orig_idx"]])
        elif label_mode == "bc":
            H_labeled = relabel_with_bc_int(d["graph"])
        else:  # degree
            H_labeled = relabel_with_degree(d["graph"])

        h = weisfeiler_lehman_hash(H_labeled)
        d["wl_hash"] = h

    # 3) Reference hash (from first original)
    if label_mode == "atom":
        ref = relabel_with_atom(dataset[group_indices[0]])
    elif label_mode == "bc":
        ref = relabel_with_bc_int(to_networkx(dataset[group_indices[0]], to_undirected=True))
    else:
        ref = relabel_with_degree(to_networkx(dataset[group_indices[0]], to_undirected=True))
    ref_hash = weisfeiler_lehman_hash(ref)

    for d in copies:
        d["survived"] = (d["wl_hash"] == ref_hash)

    # 4) Print summary
    total = len(copies)
    survived = sum(d["survived"] for d in copies)
    broken = total - survived
    add_total = sum(1 for d in copies if d["op"] == "add")
    add_survived = sum(1 for d in copies if d["op"] == "add" and d["survived"])
    rem_total = sum(1 for d in copies if d["op"] == "remove")
    rem_survived = sum(1 for d in copies if d["op"] == "remove" and d["survived"])

    print(f"Survived: {survived}, Broken: {broken}")
    print(f"ADD: {add_survived}/{add_total}, REMOVE: {rem_survived}/{rem_total}")

    # 5) Export CSV
    if csv_path:
        df = pd.DataFrame(copies)
        df.to_csv(csv_path, index=False)
        print(f"Results written to {csv_path}")

    return copies


In [11]:
if isomorphic_groups:
    group_hash, group_indices = extract_largest_group(isomorphic_groups)
    print(f"Largest WL group size: {len(group_indices)}")

    run_experiment_A("bc", group_indices, dataset, csv_path="expA_bc.csv")
    run_experiment_A("degree", group_indices, dataset, csv_path="expA_degree.csv")
    run_experiment_A("atom", group_indices, dataset, csv_path="expA_atom.csv")

Largest WL group size: 25

=== Running Experiment A with labels: bc ===
Total perturbed copies: 250
Survived: 0, Broken: 250
ADD: 0/129, REMOVE: 0/121
Results written to expA_bc.csv

=== Running Experiment A with labels: degree ===
Total perturbed copies: 250
Survived: 0, Broken: 250
ADD: 0/131, REMOVE: 0/119
Results written to expA_degree.csv

=== Running Experiment A with labels: atom ===
Total perturbed copies: 250
Survived: 70, Broken: 180
ADD: 33/126, REMOVE: 37/124
Results written to expA_atom.csv


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_bc = pd.read_csv("expA_bc.csv")
df_degree = pd.read_csv("expA_degree.csv")
df_atom = pd.read_csv("expA_atom.csv")

# Combine with labeler column
df_bc["label_mode"] = "bc"
df_degree["label_mode"] = "degree"
df_atom["label_mode"] = "atom"
df = pd.concat([df_bc, df_degree, df_atom])

# 1. Survival vs broken
survival_summary = df.groupby(["label_mode", "survived"]).size().unstack(fill_value=0)
survival_summary.plot(kind="bar", stacked=True)
plt.ylabel("Count")
plt.title("Survival vs Broken per Label Scheme")
plt.show()

# 2. Add vs remove survival
add_remove = df.groupby(["label_mode", "op", "survived"]).size().unstack(fill_value=0)
add_remove.plot(kind="bar", stacked=True)
plt.ylabel("Count")
plt.title("Add vs Remove Survival")
plt.show()
